# VIC RNA Velocity Analysis

This notebook performs two analyses on the annotated VIC cluster:

1. **Part 1** — Force-directed (ForceAtlas2) graph embedding using `scFates`
2. **Part 2** — RNA velocity estimation with `scVelo`

Each part runs in a different conda environment. **Restart the kernel and switch environments between parts.**
Outputs from Part 1 (`.h5ad` and `.npz` embeddings) are required inputs to Part 2.

# Part 1: Graph embedding (run in `scFates` environment)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os, sys

# Point to the conda R install to prevent rpy2 from crashing Jupyter
os.environ['R_HOME'] = sys.exec_prefix + "/lib/R/"

from anndata import AnnData
import numpy as np
import pandas as pd
import scanpy as sc
import scFates as scf
import palantir
import matplotlib.pyplot as plt

sc.settings.verbosity = 3
sc.settings.logfile = sys.stdout

import seaborn
seaborn.reset_orig()
%matplotlib inline
sc.set_figure_params()
scf.set_figure_pubready()

plt.rcParams['font.sans-serif'] = ['Liberation Sans']

## Parameters

In [ ]:
# Update these paths before running
data_path         = "./vics.h5ad"
output_h5ad       = "./vics_scFate.h5ad"
output_embeddings = "./graph_embeddings.npz"
output_dir        = "./"

## Load data and compute graph embedding

In [ ]:
vic_nuc = sc.read_h5ad(data_path)

# Compute neighbours in Harmony-corrected embedding space, then run a
# ForceAtlas2 force-directed layout for trajectory visualisation
sc.pp.neighbors(vic_nuc, use_rep="X_harmony")
sc.tl.draw_graph(vic_nuc)

## Fix annotation labels

In [ ]:
vic_nuc.obs["annotations_level2_readable"] = [
    f"vic{s}" for s in vic_nuc.obs["annotations_level2_readable"]
]

In [ ]:
sc.pl.draw_graph(vic_nuc, color=['annotations_level2_readable'], s=2,
                 legend_loc='on data', show=False)
plt.gca().set_aspect('equal', adjustable='datalim')
plt.show()

## Clinical group embedding density

In [ ]:
# Compute per-group kernel density on the UMAP embedding
sc.tl.embedding_density(vic_nuc, basis='umap', groupby='clinical_group')

with plt.rc_context():
    for group in ['control', 'mildmoderate', 'severe']:
        sc.pl.embedding_density(vic_nuc, basis='umap',
                                key='umap_density_clinical_group',
                                group=group, color_map='YlOrRd', show=False)
        plt.savefig(f"{output_dir}/vicn_{group}.pdf")

## Save

In [ ]:
vic_nuc.write_h5ad(output_h5ad)

# Save the ForceAtlas2 coordinates separately so they can be loaded in the
# scVelo environment
np.savez(output_embeddings, X_draw_graph_fa=vic_nuc.obsm["X_draw_graph_fa"])

# Part 2: RNA velocity (run in `scVelo` environment)

Restart the kernel and switch to the `scVelo` environment before running this section.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as pl
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata

scv.set_figure_params(style="scvelo")
pl.rcParams["figure.figsize"] = (10, 10)

# Colour palette for the 7 VIC subtypes
COLORS = ["#1B9E77", "#D95F02", "#7570B3", "#E7298A", "#0F52BA", "#E6AB02", "#000000"]

## Parameters

In [ ]:
# Update these paths before running
data_path       = "./vics.h5ad"
embeddings_path = "./graph_embeddings.npz"
loom_dir        = "./velocyto/05-velocyto"
output_h5ad     = "./velocyto/vics_scVelo.h5ad"
output_dir      = "."

## Load data

In [ ]:
vic_nuc = sc.read_h5ad(data_path)

# Restore the ForceAtlas2 embedding computed in Part 1
graph_data = np.load(embeddings_path, allow_pickle=True)
vic_nuc.obsm["X_draw_graph_fa"] = graph_data["X_draw_graph_fa"]

vic_nuc.obs["annotations_level2_readable"] = [
    f"vic{s}" for s in vic_nuc.obs["annotations_level2_readable"]
]
vic_nuc.obs["annotations_level2_readable"] = vic_nuc.obs["annotations_level2_readable"].replace(
    {f"vic{i}": f"VICn{i+1}" for i in range(7)}
)

pools = np.unique(vic_nuc.obs["pool_id"])
print("Pools:", pools)

## Load velocyto loom files and merge with expression data

In [ ]:
def transform_cell_id(cell_id, pool):
    """Convert a velocyto loom barcode to the cell name format used in the Seurat object.

    Loom barcodes are formatted as '<sample>.bam:<barcode>x'; this converts them to
    '<pool>_<barcode>-1' to match the cell names set by RenameCells() in R.
    """
    barcode = cell_id.split(':')[1].replace('x', '')
    return f"{pool}_{barcode}-1"


adata_list = []
for pool in pools:
    pool_loom = sc.read_loom(f"{loom_dir}/{pool}/velocyto/{pool}.loom")

    # Remap loom barcodes to the Seurat naming convention
    pool_loom.obs.index = list(
        map(transform_cell_id, pool_loom.obs_names, [pool] * len(pool_loom.obs_names))
    )

    # Retain only cells present in both the expression object and the loom file
    common_cells = vic_nuc.obs_names.intersection(pool_loom.obs_names)
    sub_vic  = vic_nuc[common_cells, :]
    sub_loom = pool_loom[common_cells, :]
    assert all(sub_vic.obs_names == sub_loom.obs_names)

    # Build a new AnnData: full loom gene set (needed for velocity estimation)
    # combined with the Harmony embedding and annotations from the Seurat object
    pool_adata = sc.AnnData(
        X    = sub_loom.X,
        obs  = sub_vic.obs,
        var  = sub_loom.var,
        uns  = sub_vic.uns,
        obsm = sub_vic.obsm,
        obsp = sub_vic.obsp,
    )
    pool_adata.layers['spliced']   = sub_loom.layers['spliced']
    pool_adata.layers['unspliced'] = sub_loom.layers['unspliced']
    adata_list.append(pool_adata)
    print(pool_adata)

## Concatenate pools

In [ ]:
for adata in adata_list:
    adata.var_names_make_unique()
adatas = sc.concat(adata_list, axis=0, join='outer')

## RNA velocity estimation

In [ ]:
# Recompute neighbours on the merged object (gene space differs from the original)
sc.pp.neighbors(adatas, use_rep="X_harmony")

# Deterministic scVelo model: moments -> velocity -> velocity graph -> latent time
scv.pp.moments(adatas, n_pcs=None, n_neighbors=None)
scv.tl.recover_dynamics(adatas)
scv.tl.velocity(adatas, mode='deterministic')
scv.tl.velocity_graph(adatas)
scv.tl.recover_latent_time(adatas)

## Plot and save

In [ ]:
# Preview velocity stream in the notebook
pl.rcParams["figure.figsize"] = (5, 5)
scv.pl.velocity_embedding_stream(
    adatas,
    basis="X_draw_graph_fa",
    color="annotations_level2_readable",
    title='VIC',
    fontsize=20,
    min_mass=2,
    palette=COLORS,
    legend_loc='right',
)

In [ ]:
# Save figures
with pl.rc_context({'figure.figsize': (4, 4)}):
    scv.pl.velocity_embedding_stream(
        adatas,
        basis="X_draw_graph_fa",
        color="annotations_level2_readable",
        min_mass=2,
        palette=COLORS,
        show=False,
        fontsize=0,
        legend_loc='right',
        title='',
    )
    pl.savefig(f"{output_dir}/vicn_scVelo.pdf")
    pl.savefig(f"{output_dir}/vicn_scVelo.svg", format="svg", bbox_inches='tight')

## Save

In [ ]:
adatas.write_h5ad(output_h5ad)